## Session 135 and 136

# Simple GenAI app using OpenAI

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')

In [3]:
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')

In [4]:
# use triple quotes for strings that has multiple lines 
tonu_text = """ Tonu James is a technology professional with experience in software development, computer science, artificial intelligence, and teaching. He holds a B.Tech in Computer Science and Engineering and an M.Tech in Image Processing, with expertise in computer vision, machine learning, Python, PyTorch, OpenCV, YOLO, and Edge AI deployment.
His academic and research interests include AI fairness, pedestrian detection, and human-centered artificial intelligence. He has worked on projects involving Raspberry Pi, ONNX Runtime, GStreamer, Docker, and MLOps.
Tonu is currently developing his skills as a full-stack AI developer, learning HTML, CSS, JavaScript, React, Python, Generative AI, and agentic AI. He is working on a customized Tour Management System and exploring RAG applications.
He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and teaching. Their shared professional interests include technology, continuous learning, and career development. Specific details about their wider family and personal life have not been established."""

# Convert our Text into langchain Document Type for cretaing embedding

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

split_docs = text_splitter.create_documents([tonu_text])

/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
split_docs

[Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,'),
 Document(metadata={}, page_content='computer science, artificial intelligence, and teaching. He holds a B.Tech in Computer Science and'),
 Document(metadata={}, page_content='Science and Engineering and an M.Tech in Image Processing, with expertise in computer vision,'),
 Document(metadata={}, page_content='in computer vision, machine learning, Python, PyTorch, OpenCV, YOLO, and Edge AI deployment.'),
 Document(metadata={}, page_content='His academic and research interests include AI fairness, pedestrian detection, and human-centered'),
 Document(metadata={}, page_content='and human-centered artificial intelligence. He has worked on projects involving Raspberry Pi, ONNX'),
 Document(metadata={}, page_content='Raspberry Pi, ONNX Runtime, GStreamer, Docker, and MLOps.'),
 Document(metadata={}, page_content='Tonu is currently developing his skills as 

In [6]:
from langchain_openai import OpenAIEmbeddings , ChatOpenAI

# we use openAI embedding
embeddings= OpenAIEmbeddings(model="text-embedding-3-large")

# we use LLM gpt-4o 
llm = ChatOpenAI(model="gpt-4o")

If we do a similarity search here suing vectorDB (demonstrated down) now, you only retrieve chunks. The aim in a RAG app is to pass such retrieved relevant chunks to a chat model (like GPT or Claude) along with the user's question, so it writes a natural answer like "Tonu is married to Shari Thomas, who has a background in Human-Centered AI..."

We move towards that direction .First understand the concept of Prompt.


#### Prompt Template

 Lets say in our daily converstion with LLM like ChatGPT or Claude we give a query like 
 
 "Go through below summary of my profile and based on that only answer to the below questions .Profile_summary follows . Tonu is married to Shari. Both are intrested on GenAi.  my question is who is Shari"


Using prompt template we create a similar query for RAG where we keep placeholders for our context(profile summary in this case) ,and question . 
 
 

my_prompt = ChatPromptTemplate.from_template(
"""
Go through the summary of my profile below and, based on that only,
answer the question that follows.

profile_summary:
{profile_summary}

my question is: {question}
"""
)

In [7]:
# create promt template- a reusable prompt with placeholders which are filled at runtime

from langchain_core.prompts import ChatPromptTemplate

my_prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based only on the proided context :
<context>
{context}
</context>

Question:
{input}
"""
)

## Document Chain

The concept is as below

We have Prompt template as in previous example with placeholders for context and input.


Document chain = “Before sending that message, take my Document objects, extract their text, put that text into the prompt, then call the LLM.”

We know we have our context stored in Document Object - split_docs (check the variable split_docs defined early)

Now our prompt want {context} and we have that in split_docs as [
    Document(...),
    Document(...)
]


Something needs to convert those Document objects into text. That is where create_stuff_documents_chain() comes in.

In [8]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm,my_prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the proided context :\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-openai': '1.6.6'}}, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_o

It does not run anything yet. It only assembles a small pipeline and gives it to you as an object so that we can fill the placeholders in next step.

Now we fill the placeholders in the prompt we created. We put our question and also provide the whole context that LLM has to look into in order to get the answer to the question.

In [9]:
document_chain.invoke({
    "input": "Who is Shari" ,
    "context": split_docs

})

'Shari Thomas is married to Tonu James and has a background in Human-Centered AI, software development, and teaching.'

## Concept of Retriever 

So far, we have not used a Vector Database to store embeddings and do similarity search. We directly passed our Document objects (split_docs) as the context to the document chain.

split_docs → document_chain → LLM

In a real RAG application, we may have a large collection of documents. Instead of passing all documents directly to the LLM, we first store their embeddings in a Vector Database such as Chroma or FAISS.

When the user asks a question, we use a Retriever to search the Vector Database and find the documents that are most relevant to the question.

The important idea is:

Vector Database → stores and searches the document embeddings.

Retriever → finds the relevant Document objects for a user's question.

Document Chain → takes those retrieved documents, puts their text into {context}, and sends the prompt to the LLM.

Therefore, our current approach:

document_chain.invoke({
    "input": "Who is Shari?",
    "context": split_docs
})

can later be changed so that context comes from the Retriever, rather than directly from the complete split_docs list.

To do so first lets implement the VectorDB part - we will create both Chroma and FAISS db.


In [9]:
# ChromaDB Vector Embedding and Storing

from langchain_community.vectorstores import Chroma

chroma_db= Chroma.from_documents(split_docs,embeddings )

/tmp/ipykernel_34950/3593049717.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
# FAISS 

# Vector Embedding and Storing

from langchain_community.vectorstores import FAISS

faiss_db= FAISS.from_documents(split_docs,embeddings )

Lets do similarity search for both - this step is not necessary in this project. We do that to see what result is given by similarity search.

In [13]:
# Doing query 

my_query = """who is Shari Thomas"""
chroma_retrieved_result = chroma_db.similarity_search(my_query)
print(chroma_retrieved_result)

[Document(metadata={}, page_content='He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and'), Document(metadata={}, page_content='He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and'), Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,'), Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,')]


In [14]:
my_query = """who is Shari Thomas"""
faiss_retrieved_result = faiss_db.similarity_search(my_query)
print(faiss_retrieved_result)

[Document(metadata={}, page_content='He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and'), Document(metadata={}, page_content='He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and'), Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,'), Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,')]


Now we move ahead and pass this x   

#### 1. Retriever

A retriever is a standard interface for retrieving relevant Document objects based on a user's query. A vector database can be used underneath the retriever to perform the search.

In [10]:
# create  Vector store retriever for Chroma
chroma_retriever =chroma_db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

chroma_retrieval_chain = create_retrieval_chain(chroma_retriever,document_chain)
chroma_retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7368269b56a0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the proided context :\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), add

In [11]:
# Get the response from LLM - the  see the answer in it
chroma_llm_response = chroma_retrieval_chain.invoke({"input":"Who is Shari"})
chroma_llm_response

{'input': 'Who is Shari',
 'context': [Document(metadata={}, page_content='He is married to Shari Thomas, who has a background in Human-Centered AI, software development, and'),
  Document(metadata={}, page_content='He is working on a customized Tour Management System and exploring RAG applications.'),
  Document(metadata={}, page_content='His academic and research interests include AI fairness, pedestrian detection, and human-centered'),
  Document(metadata={}, page_content='Tonu James is a technology professional with experience in software development, computer science,')],
 'answer': 'Shari Thomas is married to Tonu James and has a background in Human-Centered AI and software development.'}

In [12]:
# get answer from response 
chroma_llm_response['answer']

'Shari Thomas is married to Tonu James and has a background in Human-Centered AI and software development.'

Note : Why we didnt use "context" here - because create_retrieval_chain and create_stuff_documents_chain are not magic that reads your mind — they are hardcoded to look for two specific keys: context and input.If your template uses those exact names, everything plugs together automatically.

Imagine if we used prompt like my_prompt = ChatPromptTemplate.from_template("""
profile_summary:
{profile_summary}
my question is: {question}
""")

Then we had to use document_chain = create_stuff_documents_chain(
    llm, my_prompt, document_variable_name="profile_summary"
)

# Learning Summary -  Summary: The Evolution of a RAG Pipeline

**Text → Chunks.** We started with one block of text about Tonu and split it into small pieces using `RecursiveCharacterTextSplitter`, producing a list of `Document` objects (`split_docs`). This exists because an LLM and an embedding model work better with small, focused pieces of text than one giant blob, and overlap between chunks prevents sentences from losing context at the cut points.

**Embeddings.** We created `OpenAIEmbeddings`, a tool that converts text into a list of numbers (a vector) capturing its meaning, so that texts with similar meaning end up with similar numbers. This alone does nothing until something stores or compares those vectors.

**Prompt Template.** We learned that a prompt template is just the message you'd normally type into ChatGPT or Claude, but with blanks (`{context}`, `{input}`) that get filled in automatically instead of typed by hand each time. This separates the fixed instruction ("answer based only on context") from the parts that change (the background text and the question).

**Document Chain.** We built a small pipeline that takes `Document` objects, joins their text into one string, drops that string into the prompt's `{context}` blank, sends the finished prompt to the LLM, and extracts plain text from the LLM's reply. We tested it manually first, handing it `split_docs` directly, with no database involved, to confirm this core "stuff documents into a prompt and ask the LLM" step worked on its own.

**The problem.** At this point we noticed a limitation: we had been pasting *all* the documents into the prompt every time. That's fine for 14 small chunks, but it breaks down with a large document collection, since you can't paste everything into one prompt. We needed a way to automatically pick only the *relevant* chunks for a given question.

**Vector Database (Chroma / FAISS).** We embedded every chunk and stored those vectors in a vector database, a system built to answer "which stored vectors are closest to this new vector?" instead of exact-match lookups. We tested this directly with `similarity_search`, confirming that a query like "who is Shari Thomas" correctly surfaced the chunk about Shari as the top result, by comparing the distance between the query's vector and each stored chunk's vector.

**Retriever.** We wrapped the vector database in a `Retriever`, a standardized adapter whose only job is "take a question, call similarity search underneath, return the relevant Document objects." It added no new logic; it just gave Chroma and FAISS the same `.invoke()` interface so either one could be plugged into a chain interchangeably.

**Retrieval Chain.** Finally, we connected the Retriever to the Document Chain using `create_retrieval_chain`. This closed the loop we'd been building toward since the "manual" document chain test: instead of us deciding which documents to pass as context, the Retriever now searches the vector database automatically for each incoming question, and hands the matching chunks straight to the Document Chain.

**What we achieved.** A single call, `retrieval_chain.invoke({"input": "Who is Shari"})`, now performs the entire pipeline automatically: it searches the vector database for the most relevant chunks of Tonu's bio, inserts them into a prompt template behind the scenes, sends that filled prompt to GPT-4o, and returns a natural-language answer, *"Shari Thomas is married to Tonu James and has a background in Human-Centered AI and software development"*, grounded only in the retrieved text rather than the model's own memory. This is a complete, working RAG (Retrieval-Augmented Generation) application, scaled down to a toy example, but built from the exact same components used in production systems.